# ¿Cuántos símbolos hacen falta para decidir?
## Monitoreo secuencial de traslape espectral en señales ópticas IQ

**Proyecto final — Sistemas de Decisión Autónoma bajo Incertidumbre**  
**Ruta:** inferencia bayesiana secuencial + política de parada SARSA  
**Escenario principal:** 32 GBd, back-to-back (`32G_B2B`)

Este notebook documenta el proceso de extremo a extremo. Cada sección presenta primero la decisión metodológica y luego el código que la implementa. Al ejecutarlo desde la raíz del repositorio, genera tablas y figuras que pueden usarse para revisar el análisis.

> SARSA se estudia como política aprendida sobre reproducciones offline de las capturas. El código no controla un enlace físico ni solicita adquisiciones nuevas.


## Mapa del notebook

1. Preparar el entorno y leer la configuración.
2. Crear y revisar el inventario de adquisiciones.
3. Explorar una captura y verificar las columnas IQ.
4. Separar adquisiciones completas por canción para evitar fuga entre símbolos.
5. Ajustar KDEs con `song1` y construir el prior de condiciones.
6. Convertir cada captura en una trayectoria de creencias por bloques.
7. Entrenar SARSA con `song2`, seleccionar parámetros con `song3` y evaluar una vez con `song4`.
8. Comparar SARSA contra ventana fija y umbral de confianza.

### Alcance controlado

Para que el primer experimento sea reproducible y manejable, se enfoca en separaciones cercanas a la frontera geométrica: **35 y 35.5 GHz**. La frontera es 35.2 GHz. Los resultados de este subconjunto no deben presentarse como desempeño sobre todas las separaciones del dataset. Además, los OSNR disponibles para 35 y 35.5 GHz no necesariamente coinciden; el OSNR se incluye como condición latente en el banco, pero la falta de OSNR emparejados limita una comparación causal limpia entre separaciones.


### Antes de ejecutar

Instala las dependencias declaradas en `requirements.txt` (incluye `pyarrow` para leer Parquet) y ejecuta el notebook desde la raíz del repositorio o desde `notebooks/`. El notebook no instala paquetes automáticamente. La generación de KDE y las trayectorias puede tomar tiempo; ejecuta primero las secciones en orden y revisa las tablas de conteo antes de entrenar SARSA.


## 1. Entorno y configuración

El notebook importa los módulos del proyecto en vez de duplicar su implementación. Así, las definiciones de inventario, carga, KDE y Bayes son las mismas que usan los scripts.

In [1]:
from pathlib import Path
import sys
import json
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

# Permite ejecutar el notebook desde la raíz del repo o desde notebooks/.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "src").is_dir() and (p / "configs/experiment.yaml").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("No encuentro la raíz del proyecto (src/ y configs/experiment.yaml).")

sys.path.insert(0, str(ROOT))
from src.data.inventory import InventoryConfig, build_inventory, summarize
from src.data.loader import load_capture, load_inventory, split_into_blocks
from src.kde.bank import build_bank
from src.bayes.update import build_prior, posterior_from_loglik, overlap_prob

DATA_DIR = ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SEED = 7
rng = np.random.default_rng(SEED)
print("Raíz del proyecto:", ROOT)
print("Carpeta de datos procesados:", PROCESSED_DIR)

Raíz del proyecto: /home/oagr/Documentos/UdeA/Semestre 2026-2/SDAI/Proyecto Final
Carpeta de datos procesados: /home/oagr/Documentos/UdeA/Semestre 2026-2/SDAI/Proyecto Final/data/processed


## 2. Parámetros del experimento

La etiqueta del proyecto es geométrica: hay traslape si la separación entre canales es menor que el ancho nominal de la señal. El costo de observación y los costos de error son parámetros de decisión; no se estiman a partir de las muestras IQ.

In [2]:
with open(ROOT / "configs/experiment.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

data_cfg = cfg["data"]
RS_GBAUD = float(data_cfg["rs_gbaud"])
ROLLOFF = float(data_cfg["rolloff"])
BOUNDARY_GHZ = RS_GBAUD * (1 + ROLLOFF)
BLOCK_SIZE = 1000
MAX_BLOCKS = 10
KDE_CENTERS = 200
BANDWIDTH = float(cfg["kde"]["bandwidth"])
FOCUS_CHSP = [35.0, 35.5]

# Costos ilustrativos. Deben justificarse o someterse a sensibilidad en el informe.
C_FN = 10.0       # costo de declarar ausencia de traslape cuando sí lo hay
C_FP = 5.0        # costo de declarar traslape cuando no lo hay
C_OBS = 0.05      # costo por observar un bloque adicional
gamma = 1.0       # horizonte finito: no descontamos el costo futuro

print(f"Ancho nominal: {BOUNDARY_GHZ:.1f} GHz")
print(f"Tamaño de bloque: {BLOCK_SIZE} símbolos; máximo: {MAX_BLOCKS} bloques")
print(f"KDE: hasta {KDE_CENTERS} centros por condición, h={BANDWIDTH}")
print(f"Costos: C_FN={C_FN}, C_FP={C_FP}, C_OBS={C_OBS}")

Ancho nominal: 35.2 GHz
Tamaño de bloque: 1000 símbolos; máximo: 10 bloques
KDE: hasta 200 centros por condición, h=0.05
Costos: C_FN=10.0, C_FP=5.0, C_OBS=0.05


### Interpretación de la etiqueta

\[
B_{nom}=R_s(1+\rho)=32(1+0.1)=35.2\;\mathrm{GHz}
\]

- `z = 1`: `ChSp < 35.2 GHz` (traslape geométrico).
- `z = 0`: `ChSp > 35.2 GHz` (sin traslape geométrico).
- `ChSp = 35.2 GHz`: etiqueta ambigua; no se fuerza una clase.

Esta etiqueta no es una medición de BER ni prueba por sí sola que exista degradación en la comunicación.


## 3. Inventario de los archivos

`build_inventory` recorre el árbol configurado, identifica escenario, separación y OSNR a partir de rutas y nombres, y asigna la etiqueta geométrica. Guardar el inventario evita volver a parsear el árbol en cada etapa.

In [4]:
inventory_cfg = InventoryConfig.from_yaml(ROOT / "configs/experiment.yaml")
inv = build_inventory(inventory_cfg)
inv.to_csv(PROCESSED_DIR / "inventory.csv", index=False)

print(summarize(inv))
display(inv.head())

Total adquisiciones: 722
  con etiqueta dura (z=0/1): 722
  ambiguas (frontera):       0
  control sin interferencia: 0

Por directorio (ChSp):
  28p5GHz      n= 4  OSNR=[26.07..31.0]  z=[1]
  29GHz        n= 3  OSNR=[20.04..29.94]  z=[1]
  29p5GHz      n=16  OSNR=[19.08..31.02]  z=[1]
  30GHz        n=27  OSNR=[19.01..31.1]  z=[1]
  30p5GHz      n=36  OSNR=[18.07..31.1]  z=[1]
  31GHz        n=37  OSNR=[19.13..31.26]  z=[1]
  31p5GHz      n=38  OSNR=[17.01..30.88]  z=[1]
  32GHz        n=39  OSNR=[19.09..30.06]  z=[1]
  32p5GHz      n=45  OSNR=[17.1..31.06]  z=[1]
  33GHz        n=52  OSNR=[15.99..31.03]  z=[1]
  33p5GHz      n=54  OSNR=[16.0..31.1]  z=[1]
  34GHz        n=56  OSNR=[16.14..31.04]  z=[1]
  34p5GHz      n=52  OSNR=[18.17..31.01]  z=[1]
  35GHz        n=49  OSNR=[17.08..31.0]  z=[1]
  35p5GHz      n=54  OSNR=[18.05..30.98]  z=[0]
  36GHz        n=57  OSNR=[16.0..30.97]  z=[0]
  37p5GHz      n=49  OSNR=[15.93..30.03]  z=[0]
  50GHz        n=54  OSNR=[14.9..29.99]  z=[0]


,capture_id,scenario,chsp_ghz,chsp_dir,osnr_db,path,is_ambiguous,is_control,label_geometrica
0,32G_B2B/28p5GHz/song3_26.07dB_0km_0dBm,32G_B2B,28.5,28p5GHz,26.07,../data/raw/raw_data/32G_B2B/28p5GHz/song3_26....,False,False,1
1,32G_B2B/28p5GHz/song2_28.11dB_0km_0dBm,32G_B2B,28.5,28p5GHz,28.11,../data/raw/raw_data/32G_B2B/28p5GHz/song2_28....,False,False,1
2,32G_B2B/28p5GHz/song4_28.11dB_0km_0dBm,32G_B2B,28.5,28p5GHz,28.11,../data/raw/raw_data/32G_B2B/28p5GHz/song4_28....,False,False,1
3,32G_B2B/28p5GHz/song4_31.0dB_0km_0dBm,32G_B2B,28.5,28p5GHz,31.00,../data/raw/raw_data/32G_B2B/28p5GHz/song4_31....,False,False,1
4,32G_B2B/29GHz/song3_20.04dB_0km_0dBm,32G_B2B,29.0,29GHz,20.04,../data/raw/raw_data/32G_B2B/29GHz/song3_20.04...,False,False,1


### Revisiones de consistencia

Antes de entrenar, comprobamos cuántas adquisiciones hay por escenario, separación y canción. Una clase ausente o una canción ausente debe corregirse antes de continuar.

In [5]:
def song_from_capture_id(capture_id):
    name = str(capture_id).rsplit("/", 1)[-1]
    return name.split("_", 1)[0]

inv["song"] = inv["capture_id"].map(song_from_capture_id)
focus_inv = inv[
    inv["scenario"].eq("32G_B2B")
    & inv["chsp_ghz"].isin(FOCUS_CHSP)
    & inv["label_geometrica"].notna()
].copy()

counts = (focus_inv.groupby(["chsp_ghz", "song"], observed=True)
          .size().unstack(fill_value=0).sort_index())
display(counts)
print("Adquisiciones en el subconjunto:", len(focus_inv))
print("OSNR distintos:", focus_inv["osnr_db"].nunique())

song,song1,song2,song3,song4
chsp_ghz,,,,
35.0,13,10,13,13
35.5,12,14,14,14


Adquisiciones en el subconjunto: 103
OSNR distintos: 29


## 4. Inspección de una captura IQ

El cargador acepta Parquet con `rx_i`/`rx_q` o CSV con `I`/`Q`. La columna `tx_sym` existe en el Parquet, pero no se entrega al agente: la decisión se basa en la señal recibida.

In [6]:
example_row = focus_inv.iloc[0]
example_path = Path(example_row["path"])
raw_example = pd.read_parquet(example_path) if example_path.suffix == ".parquet" else pd.read_csv(example_path)
print("Archivo:", example_path.relative_to(ROOT))
print("Dimensiones:", raw_example.shape)
print("Columnas:", raw_example.columns.tolist())
display(raw_example.head())

example_capture = load_capture(example_row)
print("Símbolos complejos:", len(example_capture.y))
print("Potencia media tras normalización:", np.mean(np.abs(example_capture.y) ** 2))

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip, conda, or your preferred package management tool to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip, conda, or your preferred package management tool to install the fastparquet package.

In [ ]:
n_show = min(5000, len(example_capture.y))
y_show = example_capture.y[:n_show]
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_show.real, y_show.imag, s=2, alpha=0.25)
ax.set(title=f"Constelación RX: ChSp={example_capture.chsp_ghz:g} GHz, OSNR={example_capture.osnr_db:g} dB",
       xlabel="I normalizada", ylabel="Q normalizada")
ax.set_aspect("equal")
ax.grid(alpha=0.25)
plt.show()

### Nota sobre normalización y causalidad

El cargador actual normaliza cada adquisición usando la potencia media de **toda la captura**. Esto sirve para una comparación offline de formas de constelación, pero usa información de símbolos futuros y no sería un preprocesamiento causal de un monitor en línea. Si se afirma operación en línea, debe cambiarse por una calibración fija/preambulo o eliminarse esa normalización y repetir el experimento.


## 5. Partición por adquisiciones completas

La unidad de partición es el archivo de adquisición, no el símbolo ni la ventana. Como el OSNR de algunas canciones no coincide exactamente, primero se conservan solo las condiciones `(ChSp, OSNR)` que están presentes en las cuatro canciones; así el banco de `song1` sí contiene las condiciones evaluadas después. Para el experimento SARSA reservamos roles separados: `song1` ajusta las KDE, `song2` entrena la política, `song3` permite elegir hiperparámetros y `song4` queda para la evaluación final. Esta separación evita que la política memorice exactamente las señales que formaron el banco de densidades.

In [ ]:
focus_inv["song"] = focus_inv["capture_id"].map(song_from_capture_id)
required_songs = ["song1", "song2", "song3", "song4"]
condition_sets = {
    song: set(zip(
        focus_inv.loc[focus_inv["song"].eq(song), "chsp_ghz"].astype(float),
        focus_inv.loc[focus_inv["song"].eq(song), "osnr_db"].astype(float),
    ))
    for song in required_songs
}
common_conditions = set.intersection(*(condition_sets[song] for song in required_songs))
if not common_conditions:
    raise ValueError("No hay condiciones (ChSp, OSNR) compartidas entre song1-song4.")
keep_common = focus_inv.apply(
    lambda row: (float(row["chsp_ghz"]), float(row["osnr_db"])) in common_conditions,
    axis=1,
)
focus_inv = focus_inv[keep_common].copy()
print("Condiciones completas compartidas por las cuatro canciones:", len(common_conditions))
print("Adquisiciones excluidas por falta de réplica en alguna canción:", len(inv[inv["scenario"].eq("32G_B2B") & inv["chsp_ghz"].isin(FOCUS_CHSP)]) - len(focus_inv))
train_kde = focus_inv[focus_inv["song"].eq("song1")].copy()
train_policy = focus_inv[focus_inv["song"].eq("song2")].copy()
validation = focus_inv[focus_inv["song"].eq("song3")].copy()
test = focus_inv[focus_inv["song"].eq("song4")].copy()

split_summary = pd.DataFrame({
    "KDE train (song1)": [len(train_kde)],
    "SARSA train (song2)": [len(train_policy)],
    "Validación (song3)": [len(validation)],
    "Prueba (song4)": [len(test)],
})
display(split_summary)
assert all(len(part) > 0 for part in [train_kde, train_policy, validation, test]), \
    "Falta alguna partición por canción; revisar nombres y archivos del dataset."

## 6. Ajustar una KDE por condición

La KDE aproxima la densidad de `(I,Q)` para cada condición `c=(ChSp, OSNR)`. Luego se usa esa densidad para actualizar la probabilidad de las condiciones. La KDE **no** entrega directamente la probabilidad de traslape. Aquí se ajusta exclusivamente con `song1`.

In [ ]:
labels = {}
for _, row in focus_inv.drop_duplicates(["chsp_ghz", "osnr_db"]).iterrows():
    z = None if pd.isna(row["label_geometrica"]) else int(row["label_geometrica"])
    labels[(float(row["chsp_ghz"]), float(row["osnr_db"]))] = z

bank = build_bank(
    inventory=inv,
    train_capture_ids=train_kde["capture_id"].tolist(),
    bandwidth=BANDWIDTH,
    subsample=KDE_CENTERS,
    seed=SEED,
)
print("Condiciones KDE en el banco:", len(bank.keys()))
print("Primeras condiciones:", bank.keys()[:8])

prior = build_prior(bank, labels, prior_class=0.5)
print("Suma del prior:", sum(prior.values()))
print("Prior de traslape:", sum(p for c, p in prior.items() if labels.get(c) == 1))

### Qué significa el prior balanceado

El prior asigna 50 % a cada clase y reparte esa masa entre las condiciones de la clase. Así, la cantidad de separaciones/OSNR disponibles por clase no fija automáticamente la probabilidad inicial de traslape. Las condiciones que no están en el banco de entrenamiento no participan en el posterior.


## 7. Convertir una captura en una trayectoria de creencias

En cada paso se consume el siguiente bloque, se suman sus log-densidades por condición y se calcula `b_k`. El estado inicial es el prior; no equivale a haber observado un bloque.

In [ ]:
def belief_trajectory(row, block_size=BLOCK_SIZE, max_blocks=MAX_BLOCKS):
    capture = load_capture(row)
    y = capture.y[:block_size * max_blocks]
    blocks = split_into_blocks(y, block_size, disjoint=True)

    loglik = {key: 0.0 for key in bank.keys()}
    posterior0 = {key: prior[key] for key in bank.keys() if prior[key] > 0}
    beliefs = [overlap_prob(posterior0, labels)]

    for block in blocks:
        for key, condition in bank.conditions.items():
            loglik[key] += float(np.sum(condition.log_pdf(block)))
        posterior = posterior_from_loglik(loglik, prior)
        beliefs.append(overlap_prob(posterior, labels))

    return {
        "capture_id": row["capture_id"],
        "z": int(row["label_geometrica"]),
        "beliefs": np.asarray(beliefs, dtype=float),
        "blocks_available": len(blocks),
        "symbols_per_block": block_size,
    }

def make_episodes(rows):
    episodes = []
    for _, row in rows.iterrows():
        episodes.append(belief_trajectory(row))
    return episodes

# Se generan trayectorias separadas para entrenar, validar y probar.
train_episodes = make_episodes(train_policy)
val_episodes = make_episodes(validation)
test_episodes = make_episodes(test)
print("Episodios — train/validación/test:", len(train_episodes), len(val_episodes), len(test_episodes))
print("Estados de la primera trayectoria (incluye prior):", len(train_episodes[0]["beliefs"]))

## 8. Inspeccionar la evidencia acumulada

Esta figura permite comprobar si la creencia cambia gradualmente y si las capturas de ambas clases se separan. La línea horizontal en 0.5 es solo referencia; con costos asimétricos, la decisión terminal óptima no necesariamente usa ese umbral.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_episodes = [ep for z_class in (0, 1)
                 for ep in [item for item in test_episodes if item["z"] == z_class][:5]]
for episode in plot_episodes:
    b = episode["beliefs"]
    steps = np.arange(len(b))
    ax.plot(steps * BLOCK_SIZE, b, alpha=0.75,
            color="tab:orange" if episode["z"] == 1 else "tab:green",
            label="Traslape" if episode["z"] == 1 else "Sin traslape")
ax.axhline(0.5, color="black", linestyle=":", linewidth=1)
ax.set(xlabel="Símbolos observados", ylabel="P(traslape | bloques observados)", ylim=(0, 1),
       title="Trazas de creencia en capturas reservadas")
ax.grid(alpha=0.25)
handles, labels_legend = ax.get_legend_handles_labels()
unique = dict(zip(labels_legend, handles))
ax.legend(unique.values(), unique.keys())
plt.show()

## 9. Definir el MDP aproximado para SARSA

**Estado:** `(k, bin(b_k))`, donde `k` es el número de bloques ya observados y `b_k` la probabilidad agregada de traslape.  
**Acciones:** `0 = declarar sin traslape`, `1 = declarar traslape`, `2 = observar otro bloque`.  
**Recompensa:** costo negativo de observación por bloque; al parar, costo negativo si la declaración es incorrecta.

La representación escalar `b_k` es una **aproximación** del estado de creencia. Dos posteriores completos pueden tener igual probabilidad agregada de traslape y distinta incertidumbre sobre ChSp/OSNR. Se incluye `k` y se debe comparar esta aproximación contra el umbral bayesiano; no se afirma optimalidad exacta.


In [ ]:
N_BINS = 31
N_ACTIONS = 3
ALPHA = 0.15
N_EPISODES = 1200
EPSILON_START = 0.25
EPSILON_END = 0.03

# La tabla conserva el índice del bloque para representar cuánto presupuesto queda.
Q = np.zeros((MAX_BLOCKS + 1, N_BINS, N_ACTIONS), dtype=float)

def belief_bin(b):
    return int(np.clip(np.floor(float(b) * N_BINS), 0, N_BINS - 1))

def valid_actions(k, episode):
    # Solo se puede observar si existe otro bloque en la trayectoria.
    return [0, 1, 2] if k < episode["blocks_available"] else [0, 1]

def epsilon_greedy(k, b, actions, epsilon, q_table, rng):
    if rng.random() < epsilon:
        return int(rng.choice(actions))
    values = q_table[k, belief_bin(b), actions]
    best = np.flatnonzero(np.isclose(values, values.max()))
    return int(actions[rng.choice(best)])

def terminal_reward(action, z, c_fn=C_FN, c_fp=C_FP):
    declared = int(action == 1)
    if declared == z:
        return 0.0
    return -c_fn if z == 1 else -c_fp

print("Dimensiones Q:", Q.shape)
print("Estados aproximados:", (MAX_BLOCKS + 1) * N_BINS, "; acciones:", N_ACTIONS)

## 10. Entrenar SARSA (on-policy)

SARSA usa la acción que la política realmente selecciona en el estado siguiente: `Q(s,a) ← Q(s,a) + α[r + γQ(s',a') − Q(s,a)]`. Las etiquetas se usan para calcular la recompensa terminal durante el entrenamiento offline, nunca como entrada del estado ni para elegir una acción.

In [ ]:
def train_sarsa(episodes, q_table, n_episodes=N_EPISODES, seed=SEED):
    local_rng = np.random.default_rng(seed)
    returns = []

    for ep_idx in range(n_episodes):
        epsilon = EPSILON_START + (EPSILON_END - EPSILON_START) * ep_idx / max(1, n_episodes - 1)
        episode = episodes[int(local_rng.integers(len(episodes)))]
        beliefs = episode["beliefs"]
        z = episode["z"]
        k = 0
        total_reward = 0.0
        actions = valid_actions(k, episode)
        action = epsilon_greedy(k, beliefs[k], actions, epsilon, q_table, local_rng)

        while True:
            state_bin = belief_bin(beliefs[k])
            if action in (0, 1):
                reward = terminal_reward(action, z)
                q_table[k, state_bin, action] += ALPHA * (reward - q_table[k, state_bin, action])
                total_reward += reward
                break

            # Acción 2: se observa el siguiente bloque y se paga su costo.
            reward_observe = -C_OBS
            next_k = k + 1
            next_actions = valid_actions(next_k, episode)
            next_action = epsilon_greedy(
                next_k, beliefs[next_k], next_actions, epsilon, q_table, local_rng
            )

            # Actualizar primero la acción siguiente. Si es terminal, incluye su costo de error.
            next_bin = belief_bin(beliefs[next_k])
            if next_action in (0, 1):
                terminal = terminal_reward(next_action, z)
                q_table[next_k, next_bin, next_action] += ALPHA * (
                    terminal - q_table[next_k, next_bin, next_action]
                )
                target = reward_observe + gamma * q_table[next_k, next_bin, next_action]
                q_table[k, state_bin, action] += ALPHA * (target - q_table[k, state_bin, action])
                total_reward += reward_observe + terminal
                break

            target = reward_observe + gamma * q_table[next_k, next_bin, next_action]
            q_table[k, state_bin, action] += ALPHA * (target - q_table[k, state_bin, action])
            total_reward += reward_observe
            k, action = next_k, next_action

        returns.append(total_reward)

    return q_table, np.asarray(returns)

Q, training_returns = train_sarsa(train_episodes, Q)
print("Retorno medio últimas 100 épocas:", training_returns[-100:].mean())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
window = min(50, len(training_returns))
smoothed = pd.Series(training_returns).rolling(window, min_periods=1).mean()
ax.plot(smoothed)
ax.set(xlabel="Época SARSA", ylabel=f"Retorno medio móvil ({window})",
       title="Aprendizaje de la política de parada")
ax.grid(alpha=0.25)
plt.show()

## 11. Políticas de referencia

Las comparaciones usan los mismos episodios de prueba y los mismos costos. La decisión terminal de menor costo esperado declara traslape cuando `C_FP(1-b) ≤ C_FN b`, equivalente a `b ≥ C_FP/(C_FP+C_FN)`. La política por umbral solo decide temprano cuando la evidencia supera límites de confianza; al llegar al final, se ve forzada a decidir.

In [ ]:
BAYES_DECISION_THRESHOLD = C_FP / (C_FP + C_FN)
ETA_LOW = 0.10
ETA_HIGH = 0.90
FIXED_BLOCKS = min(5, MAX_BLOCKS)

def bayes_terminal_class(b):
    return int(b >= BAYES_DECISION_THRESHOLD)

def choose_sarsa_action(episode, q_table):
    beliefs = episode["beliefs"]
    for k, b in enumerate(beliefs):
        actions = [0, 1] if k >= episode["blocks_available"] else [0, 1, 2]
        values = q_table[k, belief_bin(b), actions]
        action = int(actions[np.argmax(values)])
        if action in (0, 1):
            return action, k
    return bayes_terminal_class(beliefs[-1]), episode["blocks_available"]

def run_fixed_window(episode, n_blocks=FIXED_BLOCKS):
    k = min(n_blocks, episode["blocks_available"])
    return bayes_terminal_class(episode["beliefs"][k]), k

def run_confidence_threshold(episode, eta_low=ETA_LOW, eta_high=ETA_HIGH):
    beliefs = episode["beliefs"]
    for k, b in enumerate(beliefs):
        if b <= eta_low:
            return 0, k
        if b >= eta_high:
            return 1, k
    k = episode["blocks_available"]
    return bayes_terminal_class(beliefs[k]), k

print("Umbral terminal bayesiano:", round(BAYES_DECISION_THRESHOLD, 3))
print("Ventana fija:", FIXED_BLOCKS, "bloques")
print("Umbrales de confianza:", ETA_LOW, ETA_HIGH)

## 12. Evaluación en `song4`

`song3` selecciona la ventana fija y los umbrales de confianza por costo medio. Después se congelan esos parámetros. En `song4` se reportan FPR, FNR, promedio de bloques observados y costo medio; no se ajusta ningún parámetro con la prueba final.

In [ ]:
def evaluate_policy(episodes, policy_name, policy_fn):
    rows = []
    for episode in episodes:
        prediction, n_blocks = policy_fn(episode)
        z = episode["z"]
        false_negative = int(z == 1 and prediction == 0)
        false_positive = int(z == 0 and prediction == 1)
        decision_cost = C_FN * false_negative + C_FP * false_positive
        total_cost = decision_cost + C_OBS * n_blocks
        rows.append({
            "policy": policy_name,
            "z": z,
            "prediction": prediction,
            "blocks": n_blocks,
            "symbols": n_blocks * BLOCK_SIZE,
            "FN": false_negative,
            "FP": false_positive,
            "cost": total_cost,
        })
    result = pd.DataFrame(rows)
    n_positive = max(1, int((result["z"] == 1).sum()))
    n_negative = max(1, int((result["z"] == 0).sum()))
    summary = {
        "policy": policy_name,
        "FNR": result["FN"].sum() / n_positive,
        "FPR": result["FP"].sum() / n_negative,
        "mean_blocks": result["blocks"].mean(),
        "mean_symbols": result["symbols"].mean(),
        "mean_cost": result["cost"].mean(),
        "n_test": len(result),
    }
    return summary, result

# Selección de parámetros para las referencias usando SOLO song3.
# Se escoge el menor costo medio de validación para cada familia.
fixed_candidates = []
for n_blocks in range(1, MAX_BLOCKS + 1):
    summary, _ = evaluate_policy(
        val_episodes, f"Ventana fija ({n_blocks})",
        lambda ep, n=n_blocks: run_fixed_window(ep, n),
    )
    fixed_candidates.append(summary)
FIXED_BLOCKS = int(min(fixed_candidates, key=lambda row: row["mean_cost"])["policy"].split("(")[1].split(")")[0])

threshold_candidates = []
for eta_low, eta_high in [(0.05, 0.95), (0.10, 0.90), (0.20, 0.80), (0.30, 0.70)]:
    summary, _ = evaluate_policy(
        val_episodes, f"Umbral ({eta_low:.2f}, {eta_high:.2f})",
        lambda ep, lo=eta_low, hi=eta_high: run_confidence_threshold(ep, lo, hi),
    )
    threshold_candidates.append(summary)
best_threshold = min(threshold_candidates, key=lambda row: row["mean_cost"])["policy"]
ETA_LOW, ETA_HIGH = map(float, best_threshold.split("(")[1].rstrip(")").split(", "))

validation_table = pd.DataFrame(fixed_candidates + threshold_candidates).sort_values("mean_cost")
print(f"Parámetros seleccionados en song3: N={FIXED_BLOCKS}, eta=({ETA_LOW:.2f}, {ETA_HIGH:.2f})")
display(validation_table)

policies = {
    "SARSA": lambda ep: choose_sarsa_action(ep, Q),
    "Ventana fija": run_fixed_window,
    "Umbral confianza": run_confidence_threshold,
}
summary_rows, detailed_rows = [], []
for name, policy in policies.items():
    summary, details = evaluate_policy(test_episodes, name, policy)
    summary_rows.append(summary)
    detailed_rows.append(details)

results = pd.DataFrame(summary_rows).sort_values("mean_cost")
details = pd.concat(detailed_rows, ignore_index=True)
display(results)
results.to_csv(PROCESSED_DIR / "comparacion_politicas.csv", index=False)
details.to_csv(PROCESSED_DIR / "evaluacion_por_captura.csv", index=False)

## 13. Visualizar la comparación

El costo medio depende de los costos elegidos. Revisar FPR y FNR por separado es esencial: dos políticas pueden tener costos parecidos con riesgos muy distintos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
results.set_index("policy")[["FPR", "FNR"]].plot.bar(ax=axes[0], ylim=(0, 1), rot=20)
axes[0].set(title="Errores por clase", ylabel="Tasa", xlabel="")
results.set_index("policy")["mean_symbols"].plot.bar(ax=axes[1], rot=20, color="tab:purple")
axes[1].set(title="Símbolos observados", ylabel="Promedio", xlabel="")
for ax in axes:
    ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## 14. Lectura crítica y próximos pasos

Antes de escribir conclusiones:

1. Confirmar que el inventario contiene ambas clases y suficientes adquisiciones en cada canción.
2. Revisar el balance de clases en `song4`; si una clase no aparece, FPR/FNR no son interpretables para esa partición.
3. Ajustar `N_BINS`, `C_OBS`, `C_FN`, `C_FP` y los umbrales solo con `song3`; después congelarlos y volver a evaluar `song4` una sola vez.
4. Repetir con varias semillas de SARSA y reportar media y dispersión; esta ejecución usa una semilla reproducible.
5. Analizar dependencia temporal y calibración de `b_k` (Brier score y diagrama de confiabilidad).
6. Hacer sensibilidad al tamaño de bloque, número de centros KDE, costos y conjunto de separaciones.
7. Contrastar el estado SARSA escalar `(k, bin(b))` con una representación de posterior más rica si los datos lo permiten.
8. No afirmar que SARSA es mejor solo por observar menos símbolos: comparar a tasas de error y costos equivalentes.

### Alcance de la primera versión

Este notebook es un experimento mínimo enfocado en la zona próxima a 35.2 GHz. La evaluación es offline, los costos son supuestos explícitos y el estado tabular discretiza una creencia continua. La cantidad de adquisiciones por canción limita la generalización; los resultados deben describirse como evidencia inicial. Las 1200 épocas SARSA son replays de un conjunto pequeño de trayectorias y no equivalen a 1200 adquisiciones independientes.


## 15. Artefactos generados

Al ejecutar todas las celdas, el notebook guarda el inventario, las métricas agregadas y los resultados por captura bajo `data/processed/`. Las figuras quedan visibles dentro del notebook y pueden exportarse para el informe. No se incluyen métricas prefabricadas: los resultados deben provenir de la ejecución sobre el entorno de datos real.